In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e6/train.csv
/kaggle/input/competitions/playground-series-s6e6/test.csv


In [4]:
import pandas as pd

In [5]:
traindata=pd.read_csv("/kaggle/input/competitions/playground-series-s6e6/train.csv")
testdata=pd.read_csv("/kaggle/input/competitions/playground-series-s6e6/test.csv")

In [6]:
traindata.sample(7)
# testdata.sample()

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
565975,565975,147.945474,35.595948,22.156208,22.128839,21.671879,21.649228,20.880114,1.553962,A/F,Blue_Cloud,QSO
48161,48161,123.944986,10.936907,18.887196,18.606923,18.094804,18.264319,17.707028,1.284637,G/K,Blue_Cloud,QSO
465370,465370,12.922994,-1.122211,20.396597,18.392187,17.691470,17.601638,17.572254,0.367161,G/K,Red_Sequence,GALAXY
45179,45179,128.320063,6.906337,25.261621,23.158364,21.873868,20.384020,18.467406,4.478182,M,Red_Sequence,QSO
233393,233393,126.835235,38.912883,23.154502,20.454263,19.031339,18.033062,17.671674,0.222820,M,Red_Sequence,GALAXY
555577,555577,252.521507,39.051345,20.629195,19.176828,18.282460,18.010819,17.795355,0.015673,G/K,Red_Sequence,STAR
279588,279588,35.333523,-0.977254,25.681572,23.205680,21.311397,19.975553,19.335130,0.282657,M,Red_Sequence,GALAXY


In [7]:
testdata.sample()

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
44836,622183,333.058829,2.895439,20.809597,21.063131,20.718126,20.638946,20.657228,0.607956,A/F,Blue_Cloud


In [8]:
traindata.shape

(577347, 12)

In [9]:
testdata.shape

(247435, 11)

In [10]:
traindata['spectral_type'].value_counts()

spectral_type
M      303323
A/F    122122
G/K    108546
O/B     43356
Name: count, dtype: int64

In [11]:
traindata['galaxy_population'].value_counts()

galaxy_population
Red_Sequence    319565
Blue_Cloud      257782
Name: count, dtype: int64

In [12]:
traindata['class'].value_counts()

class
GALAXY    377480
QSO       117143
STAR       82724
Name: count, dtype: int64

In [13]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()

In [14]:
traindata['spectral_type']=le.fit_transform(traindata['spectral_type'])
traindata['galaxy_population']=le.fit_transform(traindata['galaxy_population'])
traindata['class']=le.fit_transform(traindata['class'])

In [15]:
testdata['spectral_type']=le.fit_transform(testdata['spectral_type'])
testdata['galaxy_population']=le.fit_transform(testdata['galaxy_population'])
# testdata['class']=le.fit_transform(testdata['class'])

In [16]:
traindata.sample(7)

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
422691,422691,221.836276,36.066239,19.212573,17.345683,16.596827,15.837881,15.320398,0.221893,1,1,0
250193,250193,0.081043,2.181510,20.232793,20.206766,19.839670,19.965912,20.008897,1.491886,0,0,1
301007,301007,184.254033,18.867221,19.943956,18.585032,17.450718,17.112753,16.528572,0.354284,2,1,0
537420,537420,154.147324,17.906310,21.990140,21.750894,21.587929,21.505815,21.852486,0.334525,0,0,0
95339,95339,149.865786,16.136693,23.943617,22.988592,21.838720,21.487494,20.925220,1.152605,2,0,0
521602,521602,134.829406,56.890021,22.299961,21.534355,20.610362,20.812966,20.850445,0.477393,1,0,1
439105,439105,143.752681,17.582703,21.394874,21.160641,20.750467,20.490635,20.121765,2.725018,0,0,1


In [17]:
y_train=traindata['class']
x_train=traindata.drop(['id','class'],axis='columns')
testdata=testdata.drop(['id'],axis='columns')

In [18]:
# from sklearn.preprocessing  import MinMaxScaler

# mms=MinMaxScaler()
# traindata=mms.fit_transform(traindata)
# # testdata=mms.transform(testdata)

In [19]:
from sklearn.preprocessing  import StandardScaler

sc=StandardScaler()
x_train_scaled=sc.fit_transform(x_train)
y_train_scaled=sc.transform(testdata)

In [20]:
from tensorflow import keras
from tensorflow.keras.layers import Dense, Activation, Dropout
# from tensorflow.keras.Optimizers import Adam

2026-06-18 10:52:27.750785: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781779948.121490      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781779948.223140      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781779949.391934      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781779949.392002      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781779949.392006      58 computation_placer.cc:177] computation placer alr

In [21]:
model=keras.Sequential([
    keras.layers.Input(shape=(x_train.shape[1],)),

    Dense(128,activation='relu'),
    Dense(64,activation='relu'),
    Dense(32,activation='relu'),
    Dense(3,activation='softmax')
])

2026-06-18 10:52:48.813098: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [22]:
traindata

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,2,1,0
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,2,1,0
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,3,0,1
3,3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,2,1,0
4,4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,2,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
577342,577342,223.539288,2.503680,20.828729,18.854201,17.703108,17.190536,16.551356,0.511524,2,1,0
577343,577343,223.895970,40.769343,23.734743,22.359173,20.697865,19.180264,18.947275,0.658589,2,1,0
577344,577344,52.258927,0.671887,21.944250,21.215856,19.025966,18.772276,18.203397,0.376342,2,1,0
577345,577345,247.362248,50.659819,21.969881,21.622766,20.987575,20.930924,21.478134,2.868359,1,0,1


In [23]:
model.compile(
    optimizer='adam',
    metrics=['accuracy'],
    loss='sparse_categorical_crossentropy'
)

In [24]:
x_train.shape

(577347, 10)

In [25]:
y_train.shape

(577347,)

In [26]:
y_train.value_counts()

class
0    377480
1    117143
2     82724
Name: count, dtype: int64

In [27]:
model.fit(x_train_scaled,y_train,epochs=30)

Epoch 1/30
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 44s 2ms/step - accuracy: 0.9456 - loss: 0.1466
Epoch 2/30
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 39s 2ms/step - accuracy: 0.9523 - loss: 0.1284
Epoch 3/30
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 38s 2ms/step - accuracy: 0.9542 - loss: 0.1235
Epoch 4/30
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 38s 2ms/step - accuracy: 0.9552 - loss: 0.1207
Epoch 5/30
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 35s 2ms/step - accuracy: 0.9557 - loss: 0.1191
Epoch 6/30
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 34s 2ms/step - accuracy: 0.9561 - loss: 0.1183
Epoch 7/30
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 36s 2ms/step - accuracy: 0.9568 - loss: 0.1171
Epoch 8/30
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 36s 2ms/step - accuracy: 0.9568 - loss: 0.1165
Epoch 9/30
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 35s 2ms/step - accuracy: 0.9570 - loss: 0.1157
Epoch 10/30
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 35s 2ms/step - accuracy: 0.9574 - loss: 0.1150
Epoch 11/30
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 35s 2ms/step - accuracy: 0.9574 - loss: 0.11

In [28]:
model.fit(x_train_scaled,y_train,epochs=10)

Epoch 1/10
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 43s 2ms/step - accuracy: 0.9592 - loss: 0.1103
Epoch 2/10
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 43s 2ms/step - accuracy: 0.9595 - loss: 0.1102
Epoch 3/10
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 45s 2ms/step - accuracy: 0.9595 - loss: 0.1103
Epoch 4/10
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 44s 2ms/step - accuracy: 0.9594 - loss: 0.1103
Epoch 5/10
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 36s 2ms/step - accuracy: 0.9596 - loss: 0.1102
Epoch 6/10
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 36s 2ms/step - accuracy: 0.9596 - loss: 0.1110
Epoch 7/10
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 36s 2ms/step - accuracy: 0.9594 - loss: 0.1099
Epoch 8/10
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 36s 2ms/step - accuracy: 0.9595 - loss: 0.1095
Epoch 9/10
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 37s 2ms/step - accuracy: 0.9597 - loss: 0.1101
Epoch 10/10
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 36s 2ms/step - accuracy: 0.9596 - loss: 0.1100


In [29]:
model.fit(x_train_scaled,y_train,epochs=10)

Epoch 1/10
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 37s 2ms/step - accuracy: 0.9598 - loss: 0.1103
Epoch 2/10
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 38s 2ms/step - accuracy: 0.9596 - loss: 0.1103
Epoch 3/10
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 38s 2ms/step - accuracy: 0.9596 - loss: 0.1096
Epoch 4/10
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 38s 2ms/step - accuracy: 0.9597 - loss: 0.1106
Epoch 5/10
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 38s 2ms/step - accuracy: 0.9595 - loss: 0.1097
Epoch 6/10
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 40s 2ms/step - accuracy: 0.9595 - loss: 0.1096
Epoch 7/10
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 42s 2ms/step - accuracy: 0.9600 - loss: 0.1099
Epoch 8/10
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 35s 2ms/step - accuracy: 0.9599 - loss: 0.1096
Epoch 9/10
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 35s 2ms/step - accuracy: 0.9599 - loss: 0.1099
Epoch 10/10
18043/18043 ━━━━━━━━━━━━━━━━━━━━ 35s 2ms/step - accuracy: 0.9599 - loss: 0.1096


In [30]:
pred=model.predict(y_train_scaled)

7733/7733 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step


In [31]:
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd

original_train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e6/train.csv")

target_encoder = LabelEncoder()
target_encoder.fit(original_train['class'])

print("Properly restored target classes:", target_encoder.classes_)

pred_indices = np.argmax(pred, axis=1)

final_classes = target_encoder.inverse_transform(pred_indices)

Properly restored target classes: ['GALAXY' 'QSO' 'STAR']


In [32]:
final_classes

array(['GALAXY', 'GALAXY', 'GALAXY', ..., 'GALAXY', 'QSO', 'GALAXY'],
      shape=(247435,), dtype=object)

In [33]:
newtestdata=pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/test.csv')
idcol=newtestdata['id']

In [34]:
submission = pd.DataFrame({
    'id': idcol,
    'class': final_classes
})

submission.to_csv('submission.csv', index=False)

In [35]:
output=pd.read_csv('submission.csv')
output

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY
...,...,...
247430,824777,QSO
247431,824778,QSO
247432,824779,GALAXY
247433,824780,QSO


In [36]:
formatfile=pd.read_csv("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")
formatfile

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,GALAXY
4,577351,GALAXY
...,...,...
247430,824777,GALAXY
247431,824778,GALAXY
247432,824779,GALAXY
247433,824780,GALAXY
